# Topic Modeling – Exploratory Analysis of Classified News Articles

**Goal:** For each of the 13 predicted classes (`prediction`), train an LDA topic model with **5 topics**.
This reveals which concrete **events and subtopics** dominate within each class.

---

## Part 1 – Data Preparation, Stopwords & Topic Modeling

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, re, string, json
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from wordcloud import WordCloud

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.dpi': 120,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight'
})
sns.set_style('whitegrid')

# Set absolute working directory so plot paths work regardless of where the kernel was started
NOTEBOOK_DIR = Path('/Users/zorbeyozcan/news_articles_classification_thesis/Python/classification_pipeline/tests/topic modeling')
os.chdir(NOTEBOOK_DIR)

PLOT_DIR = NOTEBOOK_DIR / 'plots'

# Create all required subfolders (idempotent)
for sub in ['wordclouds', 'topic_bars', 'heatmaps', 'temporal', 'overview']:
    (PLOT_DIR / sub).mkdir(parents=True, exist_ok=True)

print('Imports successful')
print(f'CWD:      {os.getcwd()}')
print(f'PLOT_DIR: {PLOT_DIR}')

### 1.1 Load data

In [ ]:
CSV_PATH = '/Users/zorbeyozcan/news_articles_classification_thesis/data/articles/all_articles_classified Kopie.csv'

df = pd.read_csv(CSV_PATH, low_memory=False)
df['date_time'] = pd.to_datetime(df['date_time'], format='ISO8601', utc=True)
df['date'] = df['date_time'].dt.date

print(f'Total number of articles: {len(df):,}')
print(f'Time range: {df["date"].min()} to {df["date"].max()}')
print(f'\nClasses (prediction):')
print(df['prediction'].value_counts().to_string())

### 1.2 Rigorous German stopword list

Combination of:
- NLTK German stopwords (232 words)
- Custom extension: media terms, filler words, numbers, frequent news phrases, verbs, pronouns, conjunctions

In [ ]:
# NLTK baseline
STOPWORDS_DE = set(stopwords.words('german'))

# Extension: news-specific stopwords
CUSTOM_STOPS = {
    # Media / sources
    'dpa', 'afp', 'reuters', 'foto', 'bild', 'video', 'quelle', 'laut',
    'verlinkt', 'artikel', 'bericht', 'berichten', 'berichtet', 'berichtete',
    'nachrichten', 'nachricht', 'meldung', 'mitteilung', 'pressemitteilung',
    'redaktion', 'autor', 'autorin', 'kommentar', 'interview', 'ots',
    # Temporal filler words
    'bereits', 'derzeit', 'aktuell', 'inzwischen', 'mittlerweile', 'künftig',
    'kürzlich', 'zuletzt', 'demnach', 'bislang', 'bisher', 'seither',
    'seitdem', 'damals', 'gestern', 'heute', 'morgen', 'montag', 'dienstag',
    'mittwoch', 'donnerstag', 'freitag', 'samstag', 'sonntag', 'januar',
    'februar', 'märz', 'april', 'mai', 'juni', 'juli', 'august', 'september',
    'oktober', 'november', 'dezember', 'jahr', 'jahre', 'jahren', 'monat',
    'monate', 'monaten', 'woche', 'wochen', 'tag', 'tage', 'tagen',
    # Frequent verbs / auxiliaries
    'sagte', 'sagt', 'sagten', 'erklärt', 'erklärte', 'erklären', 'teilte',
    'betonte', 'betont', 'fordert', 'forderte', 'fordern', 'forderten',
    'zeigt', 'zeigte', 'zeigen', 'gab', 'gibt', 'geben', 'gegeben',
    'macht', 'machte', 'machen', 'gemacht', 'geht', 'ging', 'gehen',
    'gegangen', 'kommt', 'kam', 'kommen', 'gekommen', 'steht', 'stand',
    'stehen', 'gestanden', 'liegt', 'lag', 'liegen', 'gelegen',
    'wurde', 'wurden', 'worden', 'werde', 'werden', 'wird',
    'müsse', 'müssen', 'müsste', 'müssten', 'könne', 'können', 'könnte',
    'könnten', 'solle', 'sollen', 'sollte', 'sollten', 'wolle', 'wollen',
    'wollte', 'wollten', 'dürfe', 'dürfen', 'dürfte', 'dürften',
    'habe', 'haben', 'hatte', 'hatten', 'hätte', 'hätten', 'gehabt',
    'sei', 'seien', 'wäre', 'wären', 'gewesen',
    # Pronouns / articles (extended)
    'dessen', 'deren', 'denen', 'jene', 'jener', 'jenes', 'jenem',
    'jenen', 'solche', 'solcher', 'solches', 'solchem', 'solchen',
    'welche', 'welcher', 'welches', 'welchem', 'welchen',
    'einige', 'einigen', 'einiger', 'einiges', 'einigem',
    'mehrere', 'mehreren', 'mehrerer', 'verschiedene', 'verschiedenen',
    'verschiedener', 'verschiedenes',
    # Conjunctions / particles
    'dabei', 'darauf', 'darüber', 'darum', 'darunter', 'davon', 'davor',
    'dazu', 'dafür', 'dagegen', 'damit', 'danach', 'daneben', 'daraus',
    'darin', 'deshalb', 'deswegen', 'trotzdem', 'dennoch', 'jedoch',
    'allerdings', 'außerdem', 'zudem', 'überdies', 'hingegen', 'vielmehr',
    'insbesondere', 'beispielsweise', 'gewissermaßen', 'quasi',
    'etwa', 'rund', 'knapp', 'fast', 'nahezu', 'annähernd', 'circa',
    'ungefähr', 'sogar', 'lediglich', 'bloß', 'immerhin', 'mindestens',
    'zumindest', 'höchstens', 'wenigstens', 'jedenfalls', 'ohnehin',
    'gleichzeitig', 'unterdessen', 'währenddessen', 'schließlich',
    'letztlich', 'insgesamt', 'grundsätzlich', 'prinzipiell',
    'offenbar', 'offensichtlich', 'anscheinend', 'scheinbar', 'vermutlich',
    'wahrscheinlich', 'möglicherweise', 'eventuell', 'gegebenenfalls',
    'womöglich', 'tatsächlich', 'eigentlich', 'überhaupt', 'durchaus',
    'sowieso', 'eben', 'halt', 'recht', 'ziemlich', 'eher', 'etwas',
    'gerade', 'genau', 'nämlich', 'freilich', 'immerzu', 'abermals',
    # Quantities / numbers
    'prozent', 'million', 'millionen', 'milliarde', 'milliarden',
    'tausend', 'tausende', 'hundert', 'hunderte', 'dutzend', 'dutzende',
    'erste', 'ersten', 'erster', 'erstes', 'zweite', 'zweiten', 'zweiter',
    'dritte', 'dritten', 'dritter', 'vierte', 'vierten', 'fünfte', 'fünften',
    'euro', 'dollar', 'uhr',
    # News phrases
    'thema', 'themen', 'lesen', 'mehr', 'neue', 'neuen', 'neuer', 'neues',
    'neuem', 'groß', 'große', 'großen', 'großer', 'großes', 'großem',
    'gut', 'gute', 'guten', 'guter', 'gutes', 'gutem', 'schlecht',
    'lang', 'lange', 'langen', 'langer', 'langes', 'langem',
    'hoch', 'hohe', 'hohen', 'hoher', 'hohes', 'hohem',
    'teil', 'seite', 'fall', 'stelle', 'weise', 'art', 'grund', 'frage',
    'zeit', 'mal', 'ende', 'anfang', 'zufolge', 'lage', 'angaben',
    'deutsch', 'deutsche', 'deutschen', 'deutscher', 'deutsches', 'deutschem',
    'deutschland', 'bundesrepublik',
    # Web artifacts
    'http', 'https', 'www', 'html', 'com', 'org', 'pdf',
    # Other high-frequency generic words
    'menschen', 'mensch', 'land', 'länder', 'ländern', 'stadt', 'städte',
    'welt', 'leben', 'seit', 'beim', 'viel', 'viele', 'vielen', 'vieler',
    'wenig', 'wenige', 'wenigen', 'weniger',
    'ander', 'andere', 'anderen', 'anderer', 'anderes', 'anderem',
    'ganz', 'ganze', 'ganzen', 'ganzer', 'ganzes',
    'weiter', 'weitere', 'weiteren', 'weiterer', 'weiteres', 'weiterem',
    'allem', 'allen', 'aller', 'alles',
    'sogenannte', 'sogenannten', 'sogenannter', 'sogenanntes',
    'bestimmte', 'bestimmten', 'bestimmter', 'bestimmtes',
}

STOPWORDS_DE.update(CUSTOM_STOPS)
print(f'Stopword list: {len(STOPWORDS_DE)} words')

### 1.3 Text preprocessing

In [ ]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\b\d[\d.,]*\b', '', text)
    text = re.sub(r'[^a-zäöüß\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS_DE and len(t) > 2]
    return ' '.join(tokens)

print('Preprocessing running...')
df['text_clean'] = df['text'].apply(preprocess_text)
df['text_clean_len'] = df['text_clean'].str.split().str.len()

print('Preprocessing done')
print(f'  Average token count (cleaned): {df["text_clean_len"].mean():.0f}')
print(f'  Articles with <5 tokens after cleaning: {(df["text_clean_len"] < 5).sum():,}')

df_filtered = df[df['text_clean_len'] >= 5].copy()
print(f'  Articles after filtering: {len(df_filtered):,} (of {len(df):,})')

### 1.4 LDA Topic Modeling – 5 topics per class

A separate LDA model with **5 topics** is trained for each of the 13 classes.

In [ ]:
import time

N_TOPICS = 5
N_TOP_WORDS = 15
CLASSES = sorted(df_filtered['prediction'].unique())

# Resource control: ALL data, but minimal CPU load so you can keep working.
MAX_PER_CLASS = None     # No sampling - all articles are used
N_JOBS = 1               # Only 1 thread -> laptop stays responsive
MAX_FEATURES = 3000      # Vocabulary size per class
MAX_ITER = 20            # LDA iterations
BATCH_SIZE = 512         # Online batch size

# Lower CPU priority (nice) so the OS favours other processes
try:
    os.nice(10)
    print('os.nice(10) set - process runs in background')
except Exception as e:
    print(f'(nice could not be set: {e})')

# Hard thread limits (prevents numpy/sklearn from spawning extra threads)
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

print(f'\nClasses ({len(CLASSES)}):')
total_articles = 0
for i, c in enumerate(CLASSES, 1):
    n = (df_filtered['prediction'] == c).sum()
    total_articles += n
    print(f'  {i:2d}. {c}: {n:,} articles')
print(f'\nTotal: {total_articles:,} articles')

results = {}

print('\n' + '='*70)
print('LDA training starting (may take ~1h, laptop remains usable)...')
print('='*70)

t_start_all = time.time()

for cls_idx, cls in enumerate(CLASSES, 1):
    t_start = time.time()
    df_class = df_filtered[df_filtered['prediction'] == cls].copy().reset_index(drop=True)
    
    cv = CountVectorizer(
        max_df=0.85,
        min_df=5,
        max_features=MAX_FEATURES,
        stop_words=list(STOPWORDS_DE),
        ngram_range=(1, 2)
    )
    
    dtm = cv.fit_transform(df_class['text_clean'])
    
    lda = LatentDirichletAllocation(
        n_components=N_TOPICS,
        max_iter=MAX_ITER,
        learning_method='online',
        batch_size=BATCH_SIZE,
        random_state=42,
        n_jobs=N_JOBS
    )
    lda.fit(dtm)
    
    doc_topics = lda.transform(dtm)
    
    results[cls] = {
        'model': lda,
        'vectorizer': cv,
        'dtm': dtm,
        'df_class': df_class,
        'doc_topics': doc_topics,
        'feature_names': cv.get_feature_names_out()
    }
    
    elapsed = time.time() - t_start
    total_elapsed = time.time() - t_start_all
    print(f'\n{"="*70}')
    print(f'[{cls_idx}/{len(CLASSES)}] {cls} ({len(df_class):,} articles, {dtm.shape[1]:,} features) - {elapsed:.1f}s | total {total_elapsed/60:.1f}min')
    print(f'{"="*70}')
    for topic_idx, topic in enumerate(lda.components_):
        top_words = [results[cls]['feature_names'][i] for i in topic.argsort()[:-N_TOP_WORDS - 1:-1]]
        print(f'  Topic {topic_idx+1}: {", ".join(top_words)}')

print(f'\nLDA training finished! Total time: {(time.time()-t_start_all)/60:.1f} minutes')

In [ ]:
# Assign dominant topic per document
for cls in CLASSES:
    r = results[cls]
    doc_topics = r['doc_topics']
    df_cls = r['df_class']
    
    df_cls['dominant_topic'] = doc_topics.argmax(axis=1)
    df_cls['dominant_topic_prob'] = doc_topics.max(axis=1)
    
    topic_labels = {}
    for topic_idx, topic in enumerate(r['model'].components_):
        top3 = [r['feature_names'][i] for i in topic.argsort()[:-4:-1]]
        topic_labels[topic_idx] = f'Topic {topic_idx+1}: {", ".join(top3)}'
    
    df_cls['topic_label'] = df_cls['dominant_topic'].map(topic_labels)
    r['topic_labels'] = topic_labels
    r['df_class'] = df_cls

print('Dominant topics assigned')
for cls in CLASSES:
    print(f'\n--- {cls} ---')
    for t, label in results[cls]['topic_labels'].items():
        n = (results[cls]['df_class']['dominant_topic'] == t).sum()
        print(f'  {label}: {n:,} articles')

---

## Part 2 – Visualizations

### 2.1 Word clouds per class & topic

In [ ]:
wc_dir = PLOT_DIR / 'wordclouds'

for cls in CLASSES:
    r = results[cls]
    fig, axes = plt.subplots(1, N_TOPICS, figsize=(25, 5))
    fig.suptitle(f'Word Clouds - {cls}', fontsize=16, fontweight='bold', y=1.02)
    
    for topic_idx in range(N_TOPICS):
        topic = r['model'].components_[topic_idx]
        word_weights = {r['feature_names'][i]: topic[i] for i in topic.argsort()[:-30:-1]}
        
        wc = WordCloud(
            width=600, height=400,
            background_color='white',
            colormap='viridis',
            max_words=30,
            prefer_horizontal=0.7
        ).generate_from_frequencies(word_weights)
        
        axes[topic_idx].imshow(wc, interpolation='bilinear')
        axes[topic_idx].set_title(f'Topic {topic_idx+1}', fontsize=12)
        axes[topic_idx].axis('off')
    
    plt.tight_layout()
    safe_name = cls.replace('/', '_').replace(' ', '_')
    fig.savefig(wc_dir / f'wordcloud_{safe_name}.png', bbox_inches='tight')
    plt.show()
    plt.close()

print(f'Word clouds saved in {wc_dir}/')

### 2.2 Top words per topic (bar charts)

In [ ]:
bar_dir = PLOT_DIR / 'topic_bars'

for cls in CLASSES:
    r = results[cls]
    fig, axes = plt.subplots(1, N_TOPICS, figsize=(28, 5), sharey=False)
    fig.suptitle(f'Top-15 words per topic - {cls}', fontsize=16, fontweight='bold', y=1.02)
    
    colors = sns.color_palette('husl', N_TOPICS)
    
    for topic_idx in range(N_TOPICS):
        topic = r['model'].components_[topic_idx]
        top_indices = topic.argsort()[:-16:-1]
        top_words = [r['feature_names'][i] for i in top_indices]
        top_weights = topic[top_indices]
        top_weights = top_weights / top_weights.sum()
        
        axes[topic_idx].barh(range(len(top_words)), top_weights, color=colors[topic_idx], alpha=0.8)
        axes[topic_idx].set_yticks(range(len(top_words)))
        axes[topic_idx].set_yticklabels(top_words, fontsize=9)
        axes[topic_idx].set_title(f'Topic {topic_idx+1}', fontsize=12)
        axes[topic_idx].invert_yaxis()
        axes[topic_idx].set_xlabel('Weight (normalized)')
    
    plt.tight_layout()
    safe_name = cls.replace('/', '_').replace(' ', '_')
    fig.savefig(bar_dir / f'topic_bars_{safe_name}.png', bbox_inches='tight')
    plt.show()
    plt.close()

print(f'Topic bars saved in {bar_dir}/')

### 2.3 Topic-weight heatmaps

In [ ]:
hm_dir = PLOT_DIR / 'heatmaps'

# Global heatmap
all_means = []
row_labels = []

for cls in CLASSES:
    r = results[cls]
    mean_weights = r['doc_topics'].mean(axis=0)
    all_means.append(mean_weights)
    row_labels.append(cls)

heatmap_data = pd.DataFrame(
    all_means, index=row_labels,
    columns=[f'Topic {i+1}' for i in range(N_TOPICS)]
)

fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Mean weight'})
ax.set_title('Mean topic distribution per class', fontsize=14, fontweight='bold')
ax.set_ylabel('Class')
ax.set_xlabel('Topic')
plt.tight_layout()
fig.savefig(hm_dir / 'heatmap_global.png')
plt.show()
plt.close()

# Per-class heatmaps
for cls in CLASSES:
    r = results[cls]
    data = []
    topic_names = []
    
    for topic_idx in range(N_TOPICS):
        topic = r['model'].components_[topic_idx]
        top_indices = topic.argsort()[:-11:-1]
        top_words = [r['feature_names'][i] for i in top_indices]
        top_weights = topic[top_indices] / topic.sum()
        data.append(dict(zip(top_words, top_weights)))
        topic_names.append(f'Topic {topic_idx+1}')
    
    hm_df = pd.DataFrame(data, index=topic_names).fillna(0)
    
    fig, ax = plt.subplots(figsize=(16, 5))
    sns.heatmap(hm_df, annot=True, fmt='.4f', cmap='Blues', linewidths=0.5, ax=ax)
    ax.set_title(f'Topic-word weights - {cls}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    safe_name = cls.replace('/', '_').replace(' ', '_')
    fig.savefig(hm_dir / f'heatmap_{safe_name}.png')
    plt.show()
    plt.close()

print(f'Heatmaps saved in {hm_dir}/')

### 2.4 Temporal analysis - topic distribution over time

Shows how the topics within each class evolve over the observation period (January-February 2025).
This makes **events** visible that dominate in specific time windows.

In [ ]:
temp_dir = PLOT_DIR / 'temporal'

for cls in CLASSES:
    r = results[cls]
    df_cls = r['df_class'].copy()
    doc_topics = r['doc_topics']
    
    for t in range(N_TOPICS):
        df_cls[f'topic_{t}_prob'] = doc_topics[:, t]
    
    df_cls['date'] = pd.to_datetime(df_cls['date'])
    daily = df_cls.groupby('date')[[f'topic_{t}_prob' for t in range(N_TOPICS)]].mean()
    daily_smooth = daily.rolling(3, center=True, min_periods=1).mean()
    
    fig, ax = plt.subplots(figsize=(16, 6))
    colors = sns.color_palette('husl', N_TOPICS)
    
    for t in range(N_TOPICS):
        top3 = [r['feature_names'][i] for i in r['model'].components_[t].argsort()[:-4:-1]]
        label = f'T{t+1}: {", ".join(top3)}'
        ax.plot(daily_smooth.index, daily_smooth[f'topic_{t}_prob'],
                label=label, color=colors[t], linewidth=2, alpha=0.85)
    
    ax.set_title(f'Topic evolution over time - {cls}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Mean topic probability')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    plt.xticks(rotation=45)
    plt.tight_layout()
    safe_name = cls.replace('/', '_').replace(' ', '_')
    fig.savefig(temp_dir / f'temporal_{safe_name}.png', bbox_inches='tight')
    plt.show()
    plt.close()

print(f'Temporal plots saved in {temp_dir}/')

### 2.5 Stacked area charts - topic shares over time

In [ ]:
for cls in CLASSES:
    r = results[cls]
    df_cls = r['df_class'].copy()
    doc_topics = r['doc_topics']
    
    for t in range(N_TOPICS):
        df_cls[f'topic_{t}_prob'] = doc_topics[:, t]
    
    df_cls['date'] = pd.to_datetime(df_cls['date'])
    daily = df_cls.groupby('date')[[f'topic_{t}_prob' for t in range(N_TOPICS)]].mean()
    daily_smooth = daily.rolling(3, center=True, min_periods=1).mean()
    daily_norm = daily_smooth.div(daily_smooth.sum(axis=1), axis=0)
    
    fig, ax = plt.subplots(figsize=(16, 6))
    colors = sns.color_palette('husl', N_TOPICS)
    
    labels = []
    for t in range(N_TOPICS):
        top3 = [r['feature_names'][i] for i in r['model'].components_[t].argsort()[:-4:-1]]
        labels.append(f'T{t+1}: {", ".join(top3)}')
    
    ax.stackplot(daily_norm.index,
                 *[daily_norm[f'topic_{t}_prob'] for t in range(N_TOPICS)],
                 labels=labels, colors=colors, alpha=0.8)
    
    ax.set_title(f'Topic shares over time (stacked) - {cls}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Share')
    ax.set_ylim(0, 1)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    plt.xticks(rotation=45)
    plt.tight_layout()
    safe_name = cls.replace('/', '_').replace(' ', '_')
    fig.savefig(temp_dir / f'stacked_{safe_name}.png', bbox_inches='tight')
    plt.show()
    plt.close()

print(f'Stacked area charts saved in {temp_dir}/')

### 2.6 Article counts per topic over time

Shows the **absolute frequency** of articles per dominant topic and day.

In [ ]:
for cls in CLASSES:
    r = results[cls]
    df_cls = r['df_class'].copy()
    df_cls['date'] = pd.to_datetime(df_cls['date'])
    
    fig, ax = plt.subplots(figsize=(16, 6))
    colors = sns.color_palette('husl', N_TOPICS)
    
    for t in range(N_TOPICS):
        subset = df_cls[df_cls['dominant_topic'] == t]
        daily_counts = subset.groupby('date').size()
        daily_counts = daily_counts.reindex(
            pd.date_range(df_cls['date'].min(), df_cls['date'].max()),
            fill_value=0
        )
        smoothed = daily_counts.rolling(3, center=True, min_periods=1).mean()
        
        top3 = [r['feature_names'][i] for i in r['model'].components_[t].argsort()[:-4:-1]]
        label = f'T{t+1}: {", ".join(top3)}'
        ax.plot(smoothed.index, smoothed.values,
                label=label, color=colors[t], linewidth=2, alpha=0.85)
    
    ax.set_title(f'Article counts per topic - {cls}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Article count (3-day moving average)')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    plt.xticks(rotation=45)
    plt.tight_layout()
    safe_name = cls.replace('/', '_').replace(' ', '_')
    fig.savefig(temp_dir / f'counts_{safe_name}.png', bbox_inches='tight')
    plt.show()
    plt.close()

print(f'Article-count plots saved in {temp_dir}/')

### 2.7 Topic distribution per class (pie charts)

In [ ]:
overview_dir = PLOT_DIR / 'overview'

fig, axes = plt.subplots(4, 4, figsize=(22, 22))
axes = axes.flatten()
colors = sns.color_palette('husl', N_TOPICS)

for idx, cls in enumerate(CLASSES):
    r = results[cls]
    df_cls = r['df_class']
    counts = df_cls['dominant_topic'].value_counts().sort_index()
    
    labels = []
    for t in counts.index:
        top3 = [r['feature_names'][i] for i in r['model'].components_[t].argsort()[:-4:-1]]
        labels.append(f'T{t+1}\n{", ".join(top3)}')
    
    axes[idx].pie(counts.values, labels=labels, colors=colors,
                  autopct='%1.1f%%', textprops={'fontsize': 8},
                  pctdistance=0.85)
    axes[idx].set_title(cls, fontsize=11, fontweight='bold')

for idx in range(len(CLASSES), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle('Topic distribution per class (dominant topic)', fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(overview_dir / 'pie_charts_all_classes.png', bbox_inches='tight')
plt.show()
plt.close()

print('Pie charts saved')

### 2.8 Summary table - all topics

In [ ]:
summary_rows = []

for cls in CLASSES:
    r = results[cls]
    df_cls = r['df_class']
    
    for topic_idx in range(N_TOPICS):
        topic = r['model'].components_[topic_idx]
        top_words = [r['feature_names'][i] for i in topic.argsort()[:-11:-1]]
        
        n_articles = (df_cls['dominant_topic'] == topic_idx).sum()
        pct = n_articles / len(df_cls) * 100
        mean_prob = r['doc_topics'][:, topic_idx].mean()
        
        summary_rows.append({
            'Class': cls,
            'Topic': f'Topic {topic_idx+1}',
            'Top-10 words': ', '.join(top_words),
            'Articles (dominant)': n_articles,
            'Share (%)': round(pct, 1),
            'Mean prob.': round(mean_prob, 4)
        })

summary_df = pd.DataFrame(summary_rows)

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 100)

for cls in CLASSES:
    print(f'\n{"="*100}')
    print(f'  {cls}')
    print(f'{"="*100}')
    sub = summary_df[summary_df['Class'] == cls][['Topic', 'Top-10 words', 'Articles (dominant)', 'Share (%)', 'Mean prob.']]
    print(sub.to_string(index=False))

summary_df.to_csv('topic_summary_all_classes.csv', index=False)
print(f'\nSummary saved as topic_summary_all_classes.csv')

### 2.9 Example articles per topic

For each class the **3 most representative articles** per topic are shown (highest topic probability).

In [ ]:
for cls in CLASSES:
    r = results[cls]
    df_cls = r['df_class']
    
    print(f'\n{"#"*100}')
    print(f'### {cls}')
    print(f'{"#"*100}')
    
    for topic_idx in range(N_TOPICS):
        top3_words = [r['feature_names'][i] for i in r['model'].components_[topic_idx].argsort()[:-4:-1]]
        print(f'\n--- Topic {topic_idx+1}: {", ".join(top3_words)} ---')
        
        topic_probs = r['doc_topics'][:, topic_idx]
        top_article_indices = topic_probs.argsort()[-3:][::-1]
        
        for rank, art_idx in enumerate(top_article_indices, 1):
            row = df_cls.iloc[art_idx]
            prob = topic_probs[art_idx]
            headline = row['headline'][:100] if isinstance(row['headline'], str) else 'N/A'
            date = str(row['date'])
            domain = row['domain'] if isinstance(row['domain'], str) else 'N/A'
            print(f'  {rank}. [{date}] ({domain}) {headline}  [P={prob:.3f}]')

### 2.10 Overall overview - dominant topics across all classes

In [ ]:
overview_dir = PLOT_DIR / 'overview'

topic_word_matrix = []
y_labels = []

for cls in CLASSES:
    r = results[cls]
    for topic_idx in range(N_TOPICS):
        y_labels.append(f'{cls} - T{topic_idx+1}')
        topic = r['model'].components_[topic_idx]
        weights = topic[topic.argsort()[:-6:-1]] / topic.sum()
        topic_word_matrix.append(weights)

fig, ax = plt.subplots(figsize=(14, 35))

y_pos = np.arange(len(y_labels))
colors_all = []
palette = sns.color_palette('husl', N_TOPICS)
for cls in CLASSES:
    for t in range(N_TOPICS):
        colors_all.append(palette[t])

max_widths = [max(row) for row in topic_word_matrix]
ax.barh(y_pos, max_widths, color=colors_all, alpha=0.7, height=0.8)

for i, (label, weights) in enumerate(zip(y_labels, topic_word_matrix)):
    parts = label.rsplit(' - T', 1)
    cls = parts[0]
    t_idx = int(parts[1]) - 1
    r = results[cls]
    top5 = [r['feature_names'][j] for j in r['model'].components_[t_idx].argsort()[:-6:-1]]
    ax.text(max_widths[i] + 0.001, i, f'  {", ".join(top5)}', va='center', fontsize=7)

ax.set_yticks(y_pos)
ax.set_yticklabels(y_labels, fontsize=7)
ax.invert_yaxis()
ax.set_xlabel('Highest word weight (normalized)')
ax.set_title('All topics across all classes - overview', fontsize=16, fontweight='bold')
plt.tight_layout()
fig.savefig(overview_dir / 'complete_overview.png', bbox_inches='tight')
plt.show()
plt.close()

print('Overall overview saved')
print(f'\n{"="*70}')
print('ANALYSIS COMPLETE')
print(f'{"="*70}')
print(f'\nAll plots saved under: {PLOT_DIR.resolve()}/')
print(f'Summary: topic_summary_all_classes.csv')